In [1]:
import numpy as np
import pandas as pd

import scipy.stats as stats

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv('../data/boat_vs_noise_audio_products.csv')

# Chi-Square Test of Independence

## 1. Statistical Question

Is ANC availability associated with the brand of audio products?

### Variables
- Brand: boAt or Noise
- ANC: Present or Absent

In [ ]:
from scipy.stats import chi2_contingency, chi2


test_df = df[['brand', 'has_anc']].dropna().copy()

test_df = test_df[test_df['brand'].isin(['boAt', 'Noise'])]

print("Total observations:", len(test_df))

print("\nBrand counts:")
print(test_df['brand'].value_counts())

print("\nANC counts:")
print(test_df['has_anc'].value_counts())

Total observations: 518

Brand counts:
brand
boAt     390
Noise    128
Name: count, dtype: int64

ANC counts:
has_anc
False    428
True      90
Name: count, dtype: int64


## 2. Observed Frequency Table

A **contingency table** displays the observed frequency for each combination of brand and ANC availability.

These are the actual counts obtained from the dataset. We will compare them with the expected frequencies calculated under the assumption that brand and ANC availability are independent.

In [25]:
observed = pd.crosstab(
    test_df['brand'],
    test_df['has_anc']
)

print("Observed Frequency Table:")
display(observed)

print("\nRow Totals:")
print(observed.sum(axis=1))

print("\nColumn Totals:")
print(observed.sum(axis=0))

print("\nGrand Total:", observed.to_numpy().sum())

Observed Frequency Table:


has_anc,False,True
brand,,
Noise,96,32
boAt,332,58



Row Totals:
brand
Noise    128
boAt     390
dtype: int64

Column Totals:
has_anc
False    428
True      90
dtype: int64

Grand Total: 518


## 3. Hypotheses

**Null Hypothesis (H₀):** Brand and ANC availability are independent. There is no association between the two variables in the target population.

**Alternative Hypothesis (H₁):** Brand and ANC availability are associated.

This is a **Chi-Square Test of Independence**. The test determines whether the observed frequencies differ significantly from the frequencies expected under independence.

## 4. Assumption Checks

Before performing the test, we need to evaluate the following assumptions.

### 4.1 Categorical Variables

Both variables must be categorical.

- `brand` is categorical.
- `has_anc` is categorical.

### 4.2 Independent Observations

Each observation should contribute independently to the analysis. If the same underlying product is represented by multiple correlated listings, the independence assumption may be violated.

### 4.3 Adequate Expected Frequencies

Expected cell frequencies should generally be at least 5 for the standard Pearson chi-square approximation.

### 4.4 Mutually Exclusive Categories

Each observation must belong to exactly one brand category and exactly one ANC category.

### 4.5 Appropriate Sampling and Target Population

The data collection process must support the population-level conclusion. A statistical test cannot correct for biased sampling or dependent observations.

In [26]:
print("Brand categories:")
print(test_df['brand'].value_counts())

print("\nANC categories:")
print(test_df['has_anc'].value_counts())

print("\nMissing values:")
print(test_df[['brand', 'has_anc']].isna().sum())

print("\nNumber of brand categories:",
      test_df['brand'].nunique())

print("Number of ANC categories:",
      test_df['has_anc'].nunique())

Brand categories:
brand
boAt     390
Noise    128
Name: count, dtype: int64

ANC categories:
has_anc
False    428
True      90
Name: count, dtype: int64

Missing values:
brand      0
has_anc    0
dtype: int64

Number of brand categories: 2
Number of ANC categories: 2


### Checking Independence of Observations

Independence cannot be established through a simple numerical check alone. We must understand how the records were collected.

Repeated model names may represent different variants, while repeated listings of the same underlying product may introduce dependence. A repeated `product_id` alone does not prove dependence, especially if the identifier itself is unreliable.

We should therefore review the data collection process before making a strong population-level claim.

In [33]:
if 'base_model' in test_df.columns:
    print("Total observations:", len(test_df))
    print("Unique base models:", test_df['base_model'].nunique())

    print("\nMost frequent base models:")
    print(test_df['base_model'].value_counts().head(10))

## 5. Expected Frequencies

Under the null hypothesis, brand and ANC availability are independent.

The expected frequency for each cell is calculated as:

**Expected Frequency = (Row Total × Column Total) / Grand Total**

Expected frequencies represent the counts we would expect, on average, if there were no association between brand and ANC availability.

In [28]:
row_totals = observed.sum(axis=1)
column_totals = observed.sum(axis=0)
grand_total = observed.to_numpy().sum()

expected = np.outer(
    row_totals.to_numpy(),
    column_totals.to_numpy()
) / grand_total

expected_df = pd.DataFrame(
    expected,
    index=observed.index,
    columns=observed.columns
)

print("Expected Frequency Table:")
display(expected_df.round(2))

print("Minimum expected frequency:", expected.min())

Expected Frequency Table:


has_anc,False,True
brand,,
Noise,105.76,22.24
boAt,322.24,67.76


Minimum expected frequency: 22.23938223938224


In [29]:
if (expected >= 5).all():
    print("Assumption satisfied: all expected frequencies are at least 5.")
else:
    print("Warning: at least one expected frequency is below 5.")

Assumption satisfied: all expected frequencies are at least 5.


## 6. Perform the Chi-Square Test

Pearson's Chi-Square Test compares observed frequencies with expected frequencies.

The test statistic is:

**χ² = Σ (O − E)² / E**

Where:

- **O:** Observed frequency
- **E:** Expected frequency

The degrees of freedom are calculated as:

**df = (r − 1)(c − 1)**

Here, `r` is the number of rows and `c` is the number of columns in the contingency table.

We use a significance level of **α = 0.05**. Yates' continuity correction is disabled to calculate the standard Pearson chi-square statistic.

In [30]:
alpha = 0.05

chi2_stat, p_value, dof, expected_scipy = chi2_contingency(
    observed,
    correction=False
)

critical_value = chi2.ppf(1 - alpha, dof)

print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.6f}")
print(f"Critical value: {critical_value:.4f}")
print(f"Significance level: {alpha}")

Chi-square statistic: 6.8863
Degrees of freedom: 1
P-value: 0.008686
Critical value: 3.8415
Significance level: 0.05


## 7. Statistical Decision

We use the p-value approach to make the decision.

- **If p-value ≤ α:** Reject H₀.
- **If p-value > α:** Fail to reject H₀.

The critical-value approach gives the same decision: reject H₀ if the observed chi-square statistic exceeds the critical value.

A small p-value indicates that the observed difference between observed and expected frequencies would be unusual if H₀ were true.

In [31]:
if p_value <= alpha:
    print("Decision: Reject H0")
    print(
        "Conclusion: There is statistically significant evidence "
        "of an association between brand and ANC availability."
    )
else:
    print("Decision: Fail to reject H0")
    print(
        "Conclusion: There is insufficient statistical evidence "
        "of an association between brand and ANC availability."
    )

print("\nCritical-value approach:")

if chi2_stat > critical_value:
    print("Reject H0: chi-square statistic > critical value.")
else:
    print("Fail to reject H0: chi-square statistic <= critical value.")

Decision: Reject H0
Conclusion: There is statistically significant evidence of an association between brand and ANC availability.

Critical-value approach:
Reject H0: chi-square statistic > critical value.


## 8. Compare ANC Proportions by Brand

Statistical significance indicates evidence of an association, but it does not describe the practical strength or direction of that association.

We will compare the percentage of products with ANC in each brand and calculate **Cramér's V**, an effect-size measure for categorical association.

Cramér's V ranges from 0 to 1, with larger values indicating a stronger association. Its interpretation depends on the context and table dimensions.

In [32]:
anc_proportions = (
    pd.crosstab(
        test_df['brand'],
        test_df['has_anc'],
        normalize='index'
    ) * 100
)

print("Percentage distribution within each brand:")
display(anc_proportions.round(2))

n = observed.to_numpy().sum()
r, c = observed.shape

cramers_v = np.sqrt(
    chi2_stat / (n * min(r - 1, c - 1))
)

print(f"Cramer's V: {cramers_v:.4f}")

Percentage distribution within each brand:


has_anc,False,True
brand,,
Noise,75.00,25.00
boAt,85.13,14.87


Cramer's V: 0.1153


## 9. Final Conclusion

The Chi-Square Test of Independence evaluates whether brand and ANC availability are associated.

- **If p-value ≤ 0.05:** Reject H₀. The data provide statistically significant evidence of an association.
- **If p-value > 0.05:** Fail to reject H₀. The data do not provide sufficient statistical evidence of an association.

The ANC proportions and Cramér's V help describe the observed association.

### Limitations

- Association does not imply causation.
- Statistical significance does not necessarily imply practical importance.
- If the dataset contains every product in the target population, the observed association describes that dataset directly; a p-value should not automatically be interpreted as uncertainty about those already-observed records.
- Population-level inference requires an appropriate sampling process and reasonably independent observations.